# Week 9 Post-Class: Fine-Tune on YOUR Data

## What you'll do
Fine-tune a transfer learning model on a 5-class image dataset of your own choosing. This is your POC — the workflow you'll repeat in your career.

## Time required
60-90 minutes (most of which is collecting/curating data)

## Continuity
Same code structure as `week9_live_session.ipynb` — you're swapping the data, not the architecture.

## Step 1: Collect your data

Pick 5+ classes. Examples:
- Your own photos: pets, food categories, places
- Scraped from web: animal species, fashion items, plants
- Open datasets on Kaggle (search by topic that interests you)

**Minimum:** 50 images per class for training, 10 per class for test. More is better. Aim for 100+ training per class if possible.

Save in this folder structure:
```
my_data/
  train/
    class1/
      img1.jpg
      img2.jpg
      ...
    class2/
    class3/
    class4/
    class5/
  test/
    class1/
    class2/
    ...
```

Ensure: images are reasonable resolution (not 50×50), classes are visually distinguishable, training and test are disjoint (no leakage).

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt
import pathlib
from PIL import Image

keras.utils.set_random_seed(42)

DATA_DIR = './my_data'  # adjust to your folder
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

## Step 2: Load data and inspect

In [ ]:
def load_split(split):
    """Read my_data/<split>/<class>/*.{jpg,jpeg,png} into NumPy arrays — no TensorFlow."""
    root = pathlib.Path(DATA_DIR) / split
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    images, labels = [], []
    for label_idx, cls in enumerate(classes):
        paths = sorted(p for ext in ('*.jpg', '*.jpeg', '*.png') for p in (root / cls).glob(ext))
        for img_path in paths:
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
            images.append(np.asarray(img, dtype='float32'))
            labels.append(label_idx)
    return np.array(images), np.array(labels), classes

X_train, y_train, class_names = load_split('train')
X_test, y_test, _ = load_split('test')
print(f'Classes: {class_names}')
print(f'Training images: {X_train.shape[0]} | Test images: {X_test.shape[0]}')

In [ ]:
# Visualize sample images — sanity check that data loaded correctly
n = min(8, len(X_train))
sample_idx = np.random.RandomState(42).choice(len(X_train), n, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, sample_idx):
    ax.imshow(X_train[i].astype('uint8'))
    ax.set_title(class_names[y_train[i]])
plt.tight_layout(); plt.show()

## Step 3: Phase 1 — Frozen-base transfer learning

Identical pattern to the live session — just on your data.

In [ ]:
# YOUR TURN: build the frozen-base transfer model
# Hint: copy the pattern from the live session

# base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
# base_model.trainable = False

# inputs = keras.Input(shape=(160, 160, 3))
# x = preprocess_input(inputs)
# x = base_model(x, training=False)
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.Dropout(0.3)(x)
# outputs = layers.Dense(len(class_names), activation='softmax')(x)

# model = keras.Model(inputs, outputs)
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# model.summary()

In [ ]:
# YOUR TURN: train Phase 1
# phase1_history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
#                            epochs=10, batch_size=BATCH_SIZE, verbose=2)
# phase1_acc = model.evaluate(X_test, y_test, verbose=0)[1]
# print(f'Phase 1 test accuracy: {phase1_acc:.4f}')

## Step 4: Phase 2 — Fine-tune top layers + augmentation

In [ ]:
# YOUR TURN: unfreeze top 30 layers, add augmentation, recompile with low LR, train
# (Refer to the live session for the exact pattern)

## Step 5: Evaluate qualitatively

Plot predictions on test images. Look at misclassifications.

In [ ]:
# YOUR TURN: predict on a batch, plot images with predictions and ground truth

## Step 6: Reflect

Write your reflections in `Week9_Self_Assessment.md`. Key questions:

1. **What dataset did you use?** Briefly describe (5 classes, ~N images per class)
2. **What test accuracy did you achieve?** Phase 1 vs Phase 2
3. **At what data size did accuracy plateau?** Did adding more images help linearly or did returns diminish?
4. **Where did the model fail?** Look at misclassifications. Were the errors plausible?
5. **If you wanted to ship this as a POC**, what's the next thing you'd improve?

## Looking ahead to Week 10

Today you used a pretrained model. Same idea next week — but for **text** instead of images. The framework switches from Keras + `keras.applications` to Hugging Face `transformers`. The pattern (`from_pretrained`, freeze, add head, train) is fundamentally the same.

**Reminder:** create your free Hugging Face account at https://huggingface.co before next week if you haven't yet.